In [1]:
import io
import re

import pandas as pd
import requests
import yfinance as yf

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; QuantResearch/1.0)",
}

response = requests.get(url, headers=headers, timeout=60)
response.raise_for_status()

# Wrap HTML in StringIO so pandas/lxml parses document text, not a path/URL
tables = pd.read_html(io.StringIO(response.text))
df = tables[0]

tickers = df["Symbol"].tolist()
# Yahoo uses '-' for share classes (e.g. BRK-B); Wikipedia uses '.'
tickers = [str(s).replace(".", "-") for s in tickers]
print(tickers[:10], "...")
print(len(tickers))

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A'] ...
503


In [2]:
from pathlib import Path


def _project_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "Data" / "yahooFinance").is_dir():
            return candidate
    return p


out_dir = _project_root() / "Dataset" / "news" / "yahooFinance"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "yahoo_finance_news.csv"


def _news_row(ticker: str, rank: int, item: dict) -> dict:
    content = item.get("content") or {}
    provider = content.get("provider") or {}
    canonical = content.get("canonicalUrl") or {}
    return {
        "ticker": ticker,
        "rank": rank,
        "news_id": item.get("id"),
        "title": content.get("title"),
        "summary": content.get("summary"),
        "pub_date": content.get("pubDate"),
        "display_time": content.get("displayTime"),
        "publisher": provider.get("displayName"),
        "url": (canonical.get("url") if isinstance(canonical, dict) else None),
    }


rows: list[dict] = []
for ticker in tickers:
    try:
        news_items = yf.Ticker(ticker).news or []
        for i, item in enumerate(news_items[:10]):
            if isinstance(item, dict):
                rows.append(_news_row(ticker, i + 1, item))
    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")

df_news = pd.DataFrame(rows)
df_news.to_csv(out_csv, index=False)
print(f"Wrote {len(df_news)} rows to {out_csv}")


VST: Failed to retrieve the news and received faulty response instead.
VMC: Failed to retrieve the news and received faulty response instead.
WRB: Failed to retrieve the news and received faulty response instead.
GWW: Failed to retrieve the news and received faulty response instead.
WAB: Failed to retrieve the news and received faulty response instead.
WMT: Failed to retrieve the news and received faulty response instead.
DIS: Failed to retrieve the news and received faulty response instead.
WBD: Failed to retrieve the news and received faulty response instead.
WM: Failed to retrieve the news and received faulty response instead.
WAT: Failed to retrieve the news and received faulty response instead.
WEC: Failed to retrieve the news and received faulty response instead.
WFC: Failed to retrieve the news and received faulty response instead.
WELL: Failed to retrieve the news and received faulty response instead.
WST: Failed to retrieve the news and received faulty response instead.
WDC: F

Wrote 4760 rows to /home/mandesko/quant_project/Dataset/news/yahooFinance/yahoo_finance_news.csv
